# Session 2: rate laws and the effect of temperature and catalysts
**A. Real-world problem.** A drug tablet, a paint, a pot of stew: someone has to decide how
long a reaction needs. For that we need an equation for the rate, a way to tell from data what
kind of reaction it is, and the effect of temperature and of a catalyst on k. (General
education only; real process design needs measured data and safety review.)

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
R = 8.314  # gas constant, J/(mol K)

## Step 1: rate = k [A]^n
The *order* n says how strongly the rate depends on concentration. Solving d[A]/dt = -k [A]^n:

    n = 1:  [A] = A0 exp(-k t)            half-life = ln 2 / k  (same for any A0)
    n = 2:  [A] = A0 / (1 + k A0 t)       half-life = 1 / (k A0) (longer when A0 is small)

Units of k: 1/s for n = 1 and L/(mol s) for n = 2.

In [ ]:
def conc_first(t, A0, k):
    return A0 * np.exp(-k * np.asarray(t, dtype=float))


def conc_second(t, A0, k):
    return A0 / (1.0 + k * A0 * np.asarray(t, dtype=float))


def half_life(order, A0, k):
    if order == 1:
        return float(np.log(2.0) / k)
    return float(1.0 / (k * A0))


def time_to_conversion(order, A0, k, x):
    """Time (s) for a fraction x (0 < x < 1) of A to react."""
    if order == 1:
        return float(np.log(1.0 / (1.0 - x)) / k)
    return float(x / (k * A0 * (1.0 - x)))

## Step 2: read the order from a plot
Straight lines are easy to recognise:
* first order: **ln [A] against t** is a straight line (slope -k);
* second order: **1/[A] against t** is a straight line (slope +k).
`order_from_data` fits both lines by least squares and picks the straighter one (larger R^2).

In [ ]:
def r_squared(x, y):
    slope, icpt = np.polyfit(x, y, 1)
    resid = y - (slope * x + icpt)
    return float(1.0 - resid.var() / y.var())


def order_from_data(t, conc):
    t, conc = np.asarray(t, float), np.asarray(conc, float)
    r1 = r_squared(t, np.log(conc))
    r2 = r_squared(t, 1.0 / conc)
    return (1 if r1 >= r2 else 2), r1, r2

## Step 3: temperature and catalysts through k = A exp(-Ea / R T)
The Arrhenius form: only a small share of collisions have energy above the activation energy
Ea, and that share grows steeply with temperature T (in kelvin). A catalyst offers another
route with a lower Ea, so it multiplies k by exp(dEa / (R T)), where dEa is the drop in Ea.
(Fitting Ea from data properly is the job of the openarrhenius repo.)

In [ ]:
def arrhenius_k(A_factor, Ea, T):
    """k = A exp(-Ea / (R T)); Ea in J/mol, T in K."""
    return A_factor * np.exp(-Ea / (R * T))


def k_at_temperature(k_ref, Ea, T, T_ref=300.0):
    """k at T from a known k at T_ref (the pre-exponential factor cancels)."""
    return k_ref * np.exp(-Ea / R * (1.0 / T - 1.0 / T_ref))


def catalyst_speedup(delta_Ea, T):
    """Factor by which k grows when the activation energy drops by delta_Ea (J/mol) at T (K)."""
    return float(np.exp(delta_Ea / (R * T)))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    t = np.linspace(0, 120, 200)
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
    c1, c2 = conc_first(t, 1.0, 0.03), conc_second(t, 1.0, 0.03)
    axes[0].plot(t, c1, label="first order, k = 0.03 1/s")
    axes[0].plot(t, c2, label="second order, k = 0.03 L/(mol s)")
    axes[0].set_xlabel("time (s)")
    axes[0].set_ylabel("[A] (mol/L)")
    axes[0].legend(fontsize=7)
    axes[1].plot(t, np.log(c1), color="tab:blue")
    axes[1].plot(t, np.log(c2), color="tab:orange", ls="--")
    axes[1].set_xlabel("time (s)")
    axes[1].set_ylabel("ln [A] (no unit)")
    axes[1].set_title("first order: straight", fontsize=9)
    axes[2].plot(t, 1 / c1, color="tab:blue", ls="--")
    axes[2].plot(t, 1 / c2, color="tab:orange")
    axes[2].set_xlabel("time (s)")
    axes[2].set_ylabel("1/[A] (L/mol)")
    axes[2].set_title("second order: straight", fontsize=9)
    for a in axes:
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "orders_and_lines.png", dpi=120)

    T = np.linspace(280, 380, 100)
    fig2, ax2 = plt.subplots(figsize=(6.4, 4))
    for dEa, lab in ((0, "no catalyst"), (20000, "catalyst: Ea lower by 20 kJ/mol")):
        ax2.semilogy(T, k_at_temperature(0.01, 50000 - dEa, T) , label=lab)
    ax2.set_xlabel("temperature (K)")
    ax2.set_ylabel("rate constant k (1/s), log scale")
    ax2.set_title("k = A exp(-Ea / R T) with Ea = 50 kJ/mol")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "arrhenius_catalyst.png", dpi=120)

    print("half-life first order k=0.03:", round(half_life(1, 1.0, 0.03), 2), "s;  second order A0=1:", round(half_life(2, 1.0, 0.03), 2), "s; A0=0.5:", round(half_life(2, 0.5, 0.03), 2), "s")
    tt = np.linspace(0, 100, 12)
    print("order read from first-order data:", order_from_data(tt, conc_first(tt, 1.0, 0.03))[0],
          "; from second-order data:", order_from_data(tt, conc_second(tt, 1.0, 0.03))[0])
    print("k ratio 310 K / 300 K for Ea = 50 kJ/mol:", round(float(k_at_temperature(1.0, 50000, 310)), 3))
    print("k ratio 350 K / 300 K for Ea = 50 kJ/mol:", round(float(k_at_temperature(1.0, 50000, 350)), 1))
    print("catalyst, dEa = 20 kJ/mol at 300 K: speed-up", round(catalyst_speedup(20000, 300), 0), "times")
    print("catalyst, dEa = 20 kJ/mol at 350 K: speed-up", round(catalyst_speedup(20000, 350), 0), "times")
    k300 = 0.02
    print("design: first order k=0.02 1/s, 90% at", round(time_to_conversion(1, 1.0, k300, 0.9), 1), "s")
    k310 = float(k_at_temperature(k300, 50000, 310))
    print("   at 310 K k =", round(k310, 4), "1/s ->", round(time_to_conversion(1, 1.0, k310, 0.9), 1), "s")
    print("   catalyst dEa = 5 kJ/mol at 300 K:", round(time_to_conversion(1, 1.0, k300 * catalyst_speedup(5000, 300), 0.9), 1), "s")
    print("second order A0=1 k=0.02: 90% at", round(time_to_conversion(2, 1.0, 0.02, 0.9), 1), "s; A0 = 0.1:", round(time_to_conversion(2, 0.1, 0.02, 0.9), 1), "s")

## Where this stops
Real reactions rarely have a single step. Session 3 writes whole networks of steps as a system
of differential equations and solves it numerically.

## Coding tasks
1. **Easy.** Use `order_from_data` on a list of times and concentrations you invent (first order with k = 0.05) and check it returns order 1. Then add 5 percent noise and see when it starts to get it wrong.
2. **Stretch.** Write `temperature_for_speedup(factor, Ea, T_ref)` that returns the temperature (K) giving a chosen speed-up, and check it against the 324 K of Section F, question D.
3. **Explain.** Explain why a catalyst with the same dEa speeds a reaction up by a bigger factor at 300 K than at 350 K.

## Assessment
- **A. Conceptual.** Why does a catalyst make a reaction faster but not change how much product you finally get (for a reaction that goes to completion)?
- **B. Numerical.** For k = 0.02 1/s at 300 K and Ea = 50 kJ/mol, find k at 320 K. By what factor does it grow?
- **C. Graph and simulation interpretation.** A student plots ln[A] against t and gets a curve bending upwards, while 1/[A] against t is a straight line of slope 0.05 L/(mol s). What is the order and the rate constant? With A0 = 0.5 mol/L what is the half-life?
- **D. Design and optimisation challenge.** Design: a second-order reaction with k = 0.02 L/(mol s) at 300 K and A0 = 1.0 mol/L takes 450 s to reach 90 percent. You need 100 s. Give three ways, with numbers, and discuss the trade-offs.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("both laws start at A0", abs(float(L.conc_first(0, 2.0, 0.1)) - 2.0) < 1e-12 and abs(float(L.conc_second(0, 2.0, 0.1)) - 2.0) < 1e-12))
    h1 = L.half_life(1, 1.0, 0.03)
    checks.append(("first-order half-life is ln 2 / k = 23.1 s", abs(h1 - 23.105) < 1e-2))
    checks.append(("first-order half-life does not depend on A0", abs(L.half_life(1, 5.0, 0.03) - h1) < 1e-12))
    checks.append(("second-order [A] at its half-life is A0/2", abs(float(L.conc_second(L.half_life(2, 0.5, 0.03), 0.5, 0.03)) - 0.25) < 1e-12))
    checks.append(("second-order half-life doubles when A0 is halved", abs(L.half_life(2, 0.5, 0.03) - 2 * L.half_life(2, 1.0, 0.03)) < 1e-12))
    # worked example: design problem, 90 percent conversion
    t1 = L.time_to_conversion(1, 1.0, 0.02, 0.9)
    checks.append((f"worked example: first order, k = 0.02 1/s, 90 percent takes ln(10)/k = 115.1 s (got {t1:.1f})", abs(t1 - 115.13) < 0.05))
    checks.append(("the conversion time agrees with the formula for [A]", abs(float(L.conc_first(t1, 1.0, 0.02)) - 0.1) < 1e-9))
    t2 = L.time_to_conversion(2, 1.0, 0.02, 0.9)
    checks.append(("second order, A0 = 1, k = 0.02: 90 percent takes 9/(k A0) = 450 s", abs(t2 - 450.0) < 1e-9 and abs(float(L.conc_second(t2, 1.0, 0.02)) - 0.1) < 1e-9))
    # reading the order
    tt = np.linspace(0, 100, 12)
    checks.append(("first-order data is read as order 1", L.order_from_data(tt, L.conc_first(tt, 1.0, 0.03))[0] == 1))
    checks.append(("second-order data is read as order 2", L.order_from_data(tt, L.conc_second(tt, 1.0, 0.03))[0] == 2))
    rng = np.random.default_rng(3)
    noisy = L.conc_second(tt, 1.0, 0.03) * (1 + 0.01 * rng.standard_normal(tt.size))
    checks.append(("second-order data with 1 percent noise is still read as order 2", L.order_from_data(tt, noisy)[0] == 2))
    # Arrhenius and catalyst
    ratio = float(L.k_at_temperature(1.0, 50000.0, 310.0))
    checks.append((f"Ea = 50 kJ/mol: 10 K warmer is about 1.9 times faster (got {ratio:.2f})", abs(ratio - 1.909) < 0.01))
    checks.append(("k_at_temperature returns k_ref at T = T_ref", abs(float(L.k_at_temperature(0.02, 50000.0, 300.0)) - 0.02) < 1e-15))
    s = L.catalyst_speedup(20000.0, 300.0)
    checks.append((f"hand calculation: dEa = 20 kJ/mol at 300 K gives exp(8.018) = 3037 times (got {s:.0f})", abs(s - np.exp(20000 / (8.314 * 300))) < 1e-9 and abs(s - 3037) < 1))
    checks.append(("a catalyst with dEa = 0 changes nothing", abs(L.catalyst_speedup(0.0, 300.0) - 1.0) < 1e-15))
    checks.append(("speed-up from a catalyst is smaller at higher temperature", L.catalyst_speedup(20000, 350) < L.catalyst_speedup(20000, 300)))
    a1 = float(L.arrhenius_k(1e10, 50000.0, 300.0))
    checks.append(("arrhenius_k and k_at_temperature agree", abs(float(L.arrhenius_k(1e10, 50000.0, 310.0)) / a1 - ratio) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
